In [0]:
# Inicialização de algumas constantes

catalog = "cinedata_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"


bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/landing"

# Nome das tabelas da camada
tb_infos = "tb_movies_info"
tb_financials = "tb_movies_financials"
tb_metrics = "tb_movies_metrics"
tb_credits_and_tags = "tb_credits_and_tags"
tb_reviews = "tb_movies_reviews"

# Mapeamento dos caminhos para os arquivos brutos
path_infos = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_credits_and_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_reviews = f"{landing_path}/movies_reviews.csv"


In [0]:
from pyspark.sql.functions import (
    col,
    coalesce,
    try_to_date,
    trim,
    upper,
    regexp_replace,
    when,
    row_number,
    year,
    expr
)
from pyspark.sql.window import Window


# Leitura da Bronze
df_bronze_infos = spark.table(f"{bronze_schema}.{tb_infos}")


# Remoção de duplicatas
# Mantém a versão mais recente de cada filme, com base na data de ingestão.

print("Registros antes da deduplicação:", df_bronze_infos.count())

janela_filmes = Window.partitionBy("id").orderBy(
    col("ingestion_datetime").desc_nulls_last()
)

df_silver_infos = (
    df_bronze_infos
    .withColumn("rn", row_number().over(janela_filmes))
    .filter(col("rn") == 1)
    .drop("rn")
)

print("Registros após a deduplicação:", df_silver_infos.count())


# Renomeação das colunas

df_silver_infos = (
    df_silver_infos
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgação")
)


# Conversão tolerante da data de lançamento
# Datas inválidas ou incompatíveis são convertidas para NULL.

df_silver_infos = df_silver_infos.withColumn(
    "data_lancamento",
    coalesce(
        try_to_date(col("data_lancamento"), "yyyy-MM-dd"),
        try_to_date(col("data_lancamento"), "MM-dd-yyyy"),
        try_to_date(col("data_lancamento"), "yyyy/MM/dd"),
        try_to_date(col("data_lancamento"), "dd-MM-yyyy"),
        try_to_date(col("data_lancamento"), "dd/MM/yyyy")
    )
)


# Conversão tolerante da duração
# Valores não numéricos, tornam-se NULL.

df_silver_infos = df_silver_infos.withColumn(
    "duracao_minutos",
    expr("try_cast(duracao_minutos AS INT)")
)


# Validação básica do idioma original
# Mantém códigos de idioma de duas letras; valores inválidos viram NULL.

df_silver_infos = df_silver_infos.withColumn(
    "idioma_original",
    when(
        upper(trim(col("idioma_original"))).rlike("^[A-Z]{2}$"),
        upper(trim(col("idioma_original")))
    )
)


# Normalização do status

df_silver_infos = df_silver_infos.withColumn(
    "status_filme",
    upper(
        trim(
            regexp_replace(
                regexp_replace(col("status_filme"), r"[^a-zA-Z]+", " "),
                r"\s+",
                " "
            )
        )
    )
)


# Tradução dos status conhecidos.
# Valores desconhecidos ou inválidos são classificados como Não Informado.

df_silver_infos = df_silver_infos.withColumn(
    "status_filme",
    when(col("status_filme") == "RELEASED", "Lançado")
    .when(col("status_filme") == "POST PRODUCTION", "Pós-Produção")
    .when(col("status_filme") == "IN PRODUCTION", "Em Produção")
    .when(col("status_filme") == "PLANNED", "Planejado")
    .when(col("status_filme") == "RUMORED", "Rumores")
    .when(col("status_filme") == "CANCELED", "Cancelado")
    .otherwise("Não Informado")
)


# Criação da coluna derivada de ano de lançamento

df_silver_infos = df_silver_infos.withColumn(
    "ano_lancamento",
    year(col("data_lancamento"))
)


# Validação do resultado

print("Registros na Silver:", df_silver_infos.count())

display(
    df_silver_infos.select(
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme"
    ).limit(10)
)


# Escrita da Silver

df_silver_infos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.{tb_infos}")

In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    trim,
    upper,
    regexp_replace,
    when,
    row_number,
    round,
    expr
)
from pyspark.sql.window import Window


# Leitura da Bronze

df_bronze_financials = spark.table(
    f"{bronze_schema}.{tb_financials}"
)

print(
    "Registros antes da deduplicação:",
    df_bronze_financials.count()
)


# Remoção de duplicatas

janela_financeira = Window.partitionBy("id").orderBy(
    col("ingestion_datetime").desc_nulls_last()
)

df_silver_financeiro = (
    df_bronze_financials
    .withColumn("rn", row_number().over(janela_financeira))
    .filter(col("rn") == 1)
    .drop("rn")
)

print(
    "Registros após a deduplicação:",
    df_silver_financeiro.count()
)


# Renomeação das colunas


df_silver_financeiro = (
    df_silver_financeiro
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("budget", "orcamento_usd")
    .withColumnRenamed("revenue", "receita_usd")
)


# Limpeza e conversão dos valores monetários

# Textos que representam ausência de informação.
valores_ausentes = [
    "",
    "UNKNOWN",
    "NOT KNOWN",
    "NOT AVAILABLE",
    "N/A",
    "NA",
    "NONE",
    "NULL",
    "NAN",
    "NÃO INFORMADO",
    "NAO INFORMADO",
    "DESCONHECIDO",
    "-",
    "--"
]

def normalizar_valor_monetario(df, coluna):
    valor = upper(trim(col(coluna).cast("string")))

    ausentes = [
        "", "UNKNOWN", "NOT KNOWN", "NOT AVAILABLE",
        "N/A", "NA", "NONE", "NULL", "NAN",
        "NÃO INFORMADO", "NAO INFORMADO", "DESCONHECIDO",
        "-", "--"
    ]

    multiplicador = (
        when(valor.rlike(r"K\s*$"), lit(1000))
        .when(valor.rlike(r"M\s*$"), lit(1000000))
        .when(valor.rlike(r"B\s*$"), lit(1000000000))
        .otherwise(lit(1))
    )

    # Remove o sufixo K/M/B e símbolos monetários.
    limpo = regexp_replace(valor, r"[KMB]\s*$", "")
    limpo = regexp_replace(limpo, r"[$€£¥\s]", "")
    limpo = regexp_replace(limpo, ",", "")

    # Conversão tolerante: valores inválidos viram NULL.
    numero = expr("try_cast(NULL AS DECIMAL(18,2))")
    df = df.withColumn(
        f"_valor_{coluna}",
        expr(
            f"try_cast(regexp_replace("
            f"regexp_replace(upper(trim(cast(`{coluna}` as string))), "
            f"'[KMB]\\\\s*$', ''), "
            f"'[$€£¥,\\\\s]', '') AS DECIMAL(18,2))"
        )
    )

    df = df.withColumn(
        coluna,
        when(
            valor.isin(ausentes),
            lit(None).cast("DECIMAL(18,2)")
        ).otherwise(
            (col(f"_valor_{coluna}") * multiplicador)
            .cast("DECIMAL(18,2)")
        )
    ).drop(f"_valor_{coluna}")

    return df.withColumn(
        coluna,
        when(col(coluna) > 0, col(coluna))
        .otherwise(lit(None).cast("DECIMAL(18,2)"))
    )


df_silver_financeiro = normalizar_valor_monetario(
    df_silver_financeiro, "orcamento_usd"
)
df_silver_financeiro = normalizar_valor_monetario(
    df_silver_financeiro, "receita_usd"
)

# Obtenção da cotação do dólar
# Utiliza a cotação de compra mais recente disponível.

tb_cotacao_dolar = f"{bronze_schema}.tb_cotacao_dolar"

df_cotacao = (
    spark.table(tb_cotacao_dolar)
    .filter(col("cotacao_compra").isNotNull())
    .orderBy(
        col("data_hora_cotacao").desc_nulls_last(),
        col("ingestion_datetime").desc_nulls_last()
    )
    .select(
        col("cotacao_compra")
        .cast("decimal(18,6)")
        .alias("cotacao_usd_brl")
    )
    .limit(1)
    .withColumn("_chave_cotacao", lit(1))
)

# LEFT JOIN preserva os filmes mesmo se não houver cotação.
df_silver_financeiro = (
    df_silver_financeiro
    .withColumn("_chave_cotacao", lit(1))
    .join(df_cotacao, on="_chave_cotacao", how="left")
    .drop("_chave_cotacao")
)


# Conversão de USD para BRL


df_silver_financeiro = (
    df_silver_financeiro
    .withColumn(
        "orcamento_brl",
        when(
            col("orcamento_usd").isNotNull()
            & col("cotacao_usd_brl").isNotNull(),
            round(
                col("orcamento_usd") * col("cotacao_usd_brl"),
                2
            ).cast("decimal(18,2)")
        )
    )
    .withColumn(
        "receita_brl",
        when(
            col("receita_usd").isNotNull()
            & col("cotacao_usd_brl").isNotNull(),
            round(
                col("receita_usd") * col("cotacao_usd_brl"),
                2
            ).cast("decimal(18,2)")
        )
    )
)



# Lucro em dólar e em real

df_silver_financeiro = (
    df_silver_financeiro
    .withColumn(
        "lucro_usd",
        when(
            col("orcamento_usd").isNotNull()
            & col("receita_usd").isNotNull(),
            (
                col("receita_usd") - col("orcamento_usd")
            ).cast("decimal(18,2)")
        )
    )
    .withColumn(
        "lucro_brl",
        when(
            col("orcamento_brl").isNotNull()
            & col("receita_brl").isNotNull(),
            (
                col("receita_brl") - col("orcamento_brl")
            ).cast("decimal(18,2)")
        )
    )
)


# Margem de lucro percentual
# Margem = (receita - orçamento) / receita * 100-

df_silver_financeiro = df_silver_financeiro.withColumn(
    "margem_lucro_percentual",
    when(
        col("receita_usd").isNotNull()
        & col("orcamento_usd").isNotNull()
        & (col("receita_usd") > 0),
        round(
            (
                (col("receita_usd") - col("orcamento_usd"))
                / col("receita_usd")
            ) * 100,
            2
        ).cast("decimal(8,2)")
    )
)



# Validação do resultado

print(
    "Registros na Silver financeira:",
    df_silver_financeiro.count()
)

df_silver_financeiro.printSchema()

display(
    df_silver_financeiro.select(
        "id_filme",
        "orcamento_usd",
        "receita_usd",
        "orcamento_brl",
        "receita_brl",
        "lucro_usd",
        "lucro_brl",
        "margem_lucro_percentual",
        "cotacao_usd_brl"
    ).limit(10)
)

display(
    df_silver_financeiro
    .filter(col("id_filme").isin(38700, 38258))
    .select("id_filme", "orcamento_usd", "receita_usd", "orcamento_brl")
)


# Escrita da Silver

# (
#     df_silver_financeiro
#     .drop("cotacao_usd_brl")
#     .write
#     .format("delta")
#     .mode("overwrite")
#     .option("overwriteSchema", "true")
#     .saveAsTable(f"{silver_schema}.tb_financeiro_filmes")
# )

In [0]:
from pyspark.sql.functions import (
    col, trim, upper, regexp_replace, when, row_number
)
from pyspark.sql.window import Window

# Leitura da Bronze
df_bronze_metricas = spark.table(f"{bronze_schema}.tb_movies_metrics")

print("Registros antes da deduplicação:", df_bronze_metricas.count())

# Deduplicação: manter o registro mais recente por filme
janela_metricas = Window.partitionBy("id").orderBy(
    col("ingestion_datetime").desc_nulls_last()
)

df_silver_metricas = (
    df_bronze_metricas
    .withColumn("rn", row_number().over(janela_metricas))
    .filter(col("rn") == 1)
    .drop("rn")
)

print("Registros após a deduplicação:", df_silver_metricas.count())

# Renomear colunas
df_silver_metricas = (
    df_silver_metricas
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("popularity", "popularidade")
    .withColumnRenamed("vote_average", "nota_media_tmdb")
    .withColumnRenamed("vote_count", "qtd_votos_tmdb")
    .withColumnRenamed("averageRating", "nota_media_imdb")
    .withColumnRenamed("numVotes", "qtd_votos_imdb")
)

# Popularidade: normalizar separadores e converter com segurança.
# Trata formatos como "1,234.56" e "1234,56".


popularidade_texto = trim(col("popularidade").cast("string"))

popularidade_limpa = (
    when(
        popularidade_texto.rlike(r"^-?\d+,\d+$"),
        regexp_replace(popularidade_texto, ",", ".")
    )
    .when(
        popularidade_texto.rlike(r"^-?\d+(\.\d+)?$"),
        popularidade_texto
    )
    .otherwise(None)
)

df_silver_metricas = df_silver_metricas.withColumn(
    "popularidade",
    popularidade_limpa.try_cast("double")
)

df_silver_metricas = df_silver_metricas.withColumn(
    "popularidade",
    when(col("popularidade") >= 0, col("popularidade")).otherwise(None)
)

# Notas: conversão segura e validação no intervalo [0, 10]
for coluna in ["nota_media_tmdb", "nota_media_imdb"]:
    df_silver_metricas = df_silver_metricas.withColumn(
        coluna,
        when(
            col(coluna).cast("string").try_cast("double").between(0, 10),
            col(coluna).cast("string").try_cast("double")
        ).otherwise(None)
    )

# Contagens: conversão segura para inteiro e invalidação de negativos
for coluna in ["qtd_votos_tmdb", "qtd_votos_imdb"]:
    df_silver_metricas = df_silver_metricas.withColumn(
        coluna,
        when(
            col(coluna).cast("string").try_cast("long") >= 0,
            col(coluna).cast("string").try_cast("long")
        ).otherwise(None)
    )

# Popularidade negativa deve ser NULL
df_silver_metricas = df_silver_metricas.withColumn(
    "popularidade",
    when(col("popularidade") >= 0, col("popularidade")).otherwise(None)
)

# Persistir na Silver
(
    df_silver_metricas
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_metricas_engajamento")
)

print("Tabela Silver de métricas de engajamento criada com sucesso.")
display(
    spark.table(f"{silver_schema}.tb_metricas_engajamento")
    .limit(10)
)

In [0]:
from pyspark.sql.functions import col, trim, when, lit

# Leitura da Bronze
df_bronze_avaliacoes = spark.table(f"{bronze_schema}.tb_movies_reviews")

print("Registros antes da deduplicação:", df_bronze_avaliacoes.count())

# Renomear colunas
df_silver_avaliacoes = (
    df_bronze_avaliacoes
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario")
)

# Remover avaliações integralmente duplicadas
df_silver_avaliacoes = df_silver_avaliacoes.dropDuplicates([
    "id_filme",
    "nome_usuario",
    "nota_usuario",
    "comentario_usuario"
])

print("Registros após deduplicação:", df_silver_avaliacoes.count())

# Converter nota com segurança e invalidar valores fora de 0 a 10
nota_convertida = col("nota_usuario").cast("string").try_cast("double")

df_silver_avaliacoes = df_silver_avaliacoes.withColumn(
    "nota_usuario",
    when(
        nota_convertida.between(0, 10),
        nota_convertida
    ).otherwise(lit(None).cast("double"))
)

# Substituir comentários nulos, vazios ou com apenas espaços
df_silver_avaliacoes = df_silver_avaliacoes.withColumn(
    "comentario_usuario",
    when(
        col("comentario_usuario").isNull()
        | (trim(col("comentario_usuario")) == ""),
        lit("Sem comentário")
    ).otherwise(col("comentario_usuario"))
)

# Salvar na Silver
(
    df_silver_avaliacoes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")
)

print("Tabela Silver de avaliações criada com sucesso.")

# Conferência dos primeiros registros
display(
    spark.table(f"{silver_schema}.tb_avaliacoes_usuarios").limit(10)
)

In [0]:
from pyspark.sql.functions import (
    col, trim, lower, regexp_replace, split, explode, max as spark_max
)

# Leitura da Bronze
df_bronze_generos = spark.table(f"{bronze_schema}.tb_credits_and_tags")

print("Registros na Bronze:", df_bronze_generos.count())

# Normalizar separadores e remover resíduos de formatação
generos_normalizados = regexp_replace(
    col("genres").cast("string"),
    r"[;]",
    ","
)

generos_normalizados = regexp_replace(
    generos_normalizados,
    r"""[\[\]\{\}"']""",
    ""
)

# Explodir: uma linha por gênero
df_silver_generos = (
    df_bronze_generos
    .select(
        col("id").alias("id_filme"),
        explode(split(generos_normalizados, ",")).alias("genero"),
        col("ingestion_datetime")
    )
    .withColumn("genero", trim(col("genero")))
)

# Domínio válido de gêneros
generos_validos = [
    "action",
    "adventure",
    "animation",
    "comedy",
    "crime",
    "documentary",
    "drama",
    "family",
    "fantasy",
    "history",
    "horror",
    "music",
    "mystery",
    "romance",
    "science fiction",
    "tv movie",
    "thriller",
    "war",
    "western"
]

# Remover vazios e valores fora do domínio
df_silver_generos = (
    df_silver_generos
    .filter(lower(col("genero")).isin(generos_validos))
    .dropDuplicates(["id_filme", "genero"])
)

print("Gêneros válidos após limpeza:", df_silver_generos.count())

# Salvar na Silver
(
    df_silver_generos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_generos")
)

print("Tabela Silver de gêneros criada com sucesso.")

# Conferir resultado
display(
    spark.table(f"{silver_schema}.tb_generos").limit(10)
)

In [0]:
from pyspark.sql.functions import (
    col, lit, trim, initcap, explode, split,
    regexp_replace, when
)
from functools import reduce

# Leitura da Bronze
df_bronze_creditos = spark.table(f"{bronze_schema}.tb_credits_and_tags")

# Mapeamento das colunas para os tipos de entidade
mapeamento_entidades = [
    ("cast", "Ator"),
    ("directors", "Diretor"),
    ("writers", "Roteirista"),
    ("production_companies", "Produtora")
]

# Explodir e unificar as entidades
dfs_entidades = []

for coluna_origem, tipo in mapeamento_entidades:
    df_entidade = (
        df_bronze_creditos
        .select(
            col("id").alias("id_filme"),
            explode(
                split(
                    regexp_replace(
                        col(coluna_origem).cast("string"),
                        ";",
                        ","
                    ),
                    ","
                )
            ).alias("nome_entidade")
        )
        .withColumn("nome_entidade", trim(col("nome_entidade")))
        .withColumn("nome_entidade", initcap(col("nome_entidade")))
        .withColumn("tipo_entidade", lit(tipo))
        .filter(
            col("nome_entidade").isNotNull()
            & (col("nome_entidade") != "")
            & ~col("nome_entidade").rlike(r"^\d+([.,]\d+)?$")
        )
        .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
    )

    dfs_entidades.append(df_entidade)

df_silver_pessoas_empresas = reduce(
    lambda a, b: a.unionByName(b),
    dfs_entidades
)

# Persistir na Silver
(
    df_silver_pessoas_empresas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_pessoas_empresas")
)

print("Tabela Silver de pessoas e empresas criada com sucesso.")

display(
    spark.table(f"{silver_schema}.tb_pessoas_empresas").limit(10)
)

In [0]:
from pyspark.sql.functions import (
    col, to_date, max as spark_max, min as spark_min,
    last, explode, sequence, lit, row_number
)
from pyspark.sql.window import Window

# Leitura da Bronze
df_bronze_cotacao = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

# Uma cotação por dia: manter a última cotação registrada naquele dia
janela_cotacao = Window.partitionBy(
    to_date("data_hora_cotacao")
).orderBy(col("data_hora_cotacao").desc())

df_cotacoes_diarias = (
    df_bronze_cotacao
    .withColumn("data_cotacao", to_date("data_hora_cotacao"))
    .withColumn("rn", row_number().over(janela_cotacao))
    .filter(col("rn") == 1)
    .select(
        "data_cotacao",
        col("cotacao_compra").cast("double").alias("cotacao_compra")
    )
)

# Gerar calendário contínuo entre a primeira e a última data disponível
limites = df_cotacoes_diarias.agg(
    spark_min("data_cotacao").alias("data_min"),
    spark_max("data_cotacao").alias("data_max")
).first()

df_calendario = (
    spark.range(1)
    .select(
        explode(
            sequence(
                lit(limites["data_min"]),
                lit(limites["data_max"]),
                lit(1).cast("interval day")
            )
        ).alias("data_cotacao")
    )
)

# Juntar calendário às cotações e aplicar Forward Fill
janela_forward_fill = (
    Window.orderBy("data_cotacao")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_silver_cotacao = (
    df_calendario
    .join(df_cotacoes_diarias, on="data_cotacao", how="left")
    .withColumn(
        "cotacao_compra",
        last("cotacao_compra", ignorenulls=True).over(janela_forward_fill)
    )
    .withColumnRenamed("data_cotacao", "data")
)

# Persistir na Silver
(
    df_silver_cotacao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_cotacao_dolar")
)

print("Tabela Silver de cotação do dólar criada com sucesso.")

display(
    spark.table(f"{silver_schema}.tb_cotacao_dolar")
    .orderBy(col("data").desc())
    .limit(10)
)